In [ ]:
# pip install -U langchain langchain-openai langchain-community faiss-cpu pypdf python-dotenv langsmith

import os
from dotenv import load_dotenv

from langsmith import traceable

from langchain_community.document_loaders import PyPDFLoader
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_openrouter import ChatOpenRouter
from langchain_nomic import NomicEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

os.environ['LANGSMITH_PROJECT'] = 'PDF RAG'

load_dotenv()


PDF_PATH = "islr.pdf"  

# ----------------- helpers (not traced individually) -----------------
@traceable(name="load_pdf")
def load_pdf(path: str):
    loader = PyPDFLoader(path)
    return loader.load()  # list[Document]

@traceable(name="split_documents")
def split_documents(docs, chunk_size, chunk_overlap):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap
    )
    return splitter.split_documents(docs)

@traceable(name="build_vectorstore")
def build_vectorstore(splits):
    emb = NomicEmbeddings(model="nomic-embed-text-v1.5")
    return FAISS.from_documents(splits, emb)

# ----------------- parent setup function (traced) -----------------
@traceable(name="setup_pipeline", tags=["setup"])
def setup_pipeline(pdf_path: str, chunk_size, chunk_overlap):
    # These three steps are “clubbed” under this parent function
    docs = load_pdf(pdf_path)
    splits = split_documents(docs, chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    vs = build_vectorstore(splits)
    return vs

# ----------------- model, prompt, and run -----------------
llm = ChatOpenRouter(model="gpt-4o-mini", temperature=0.2, top_p=0.3)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer ONLY from the provided context. If not found, say you don't know."),
    ("human", "Question: {question}\n\nContext:\n{context}")
])

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

# ----------------- one top-level (root) run -----------------
@traceable(name="pdf_rag_full_run")
def setup_pipeline_and_query(pdf_path: str, question: str):
    # Parent setup run (child of root)
    vectorstore = setup_pipeline(pdf_path, chunk_size=2000, chunk_overlap=180)

    retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 3})

    parallel = RunnableParallel({
        "context": retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough(),
    })

    chain = parallel | prompt | llm | StrOutputParser()

    # This LangChain run stays under the same root (since we're inside this traced function)
    lc_config = {"run_name": "pdf_rag_query"}
    return chain.invoke(question, config=lc_config)

# ----------------- CLI -----------------
print("PDF RAG ready. Ask a question (or Ctrl+C to exit).")
q = input("\nQ: ").strip()
ans = setup_pipeline_and_query(PDF_PATH, q)
print("\nAns:", ans)

PDF RAG ready. Ask a question (or Ctrl+C to exit).



Ans: The `plot()` function in R is the primary way to visualize data. It can create various types of plots, with the most common being scatterplots. For example, using `plot(x, y)` will produce a scatterplot of the values in `x` against the values in `y`. 

The `plot()` function accepts many additional options to customize the appearance of the plot. For instance, you can add labels to the axes using the `xlab` and `ylab` arguments, and you can set a title for the plot using the `main` argument. To explore more details about the function, you can type `?plot` in the R console. 

An example usage would be:
```R
> x = rnorm(100)
> y = rnorm(100)
> plot(x, y)
> plot(x, y, xlab="this is the x-axis", ylab="this is the y-axis", main="Plot of X vs Y")
```
